# 📥 SnowGhost Breakers Data Loader

**Purpose:** Bulk load ghost detection data into Snowflake

**Based on:** https://github.com/tspannhw/AIM-Ghosts/blob/main/standardghostload.ipynb

**Features:**
- Load sightings from CSV/JSON
- Bulk image upload to stages
- Batch evidence processing
- Data validation and quality checks
- Progress tracking and error handling


## 🔧 Setup & Configuration


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, lit, when
from datetime import datetime, timedelta
import json
from pathlib import Path
import uuid

print("✅ Libraries imported successfully")


In [ ]:
# Create Snowflake session
from snowflake.snowpark.context import get_active_session

# For Snowflake Notebooks use get_active_session
session = get_active_session()
print("Connected to Snowflake")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")


## 📊 Data Validation Functions


In [ ]:
def validate_coordinates(lat, lon):
    """Validate latitude and longitude."""
    return (-90 <= lat <= 90) and (-180 <= lon <= 180)

def validate_activity_level(level):
    """Validate paranormal activity level (1-10)."""
    return 1 <= level <= 10

def validate_temperature(temp_c):
    """Validate temperature in Celsius."""
    return -50 <= temp_c <= 50

def validate_emf(emf):
    """Validate EMF reading."""
    return 0 <= emf <= 100

print("✅ Validation functions defined")


## 📥 Load 1: Ghost Registry Data


In [ ]:
# Generate sample ghost data
ghosts_data = []
ghost_types = ['Poltergeist', 'Phantom', 'Wraith', 'Apparition', 'Specter']
threat_levels = ['Low', 'Medium', 'High', 'Extreme']
frequencies = ['Rare', 'Occasional', 'Frequent', 'Constant']

# Generate varied origin stories
origins = [
    'Tragic death in Victorian era',
    'Unfinished business from past life',
    'Violent death during war',
    'Murdered in this location',
    'Died protecting loved ones',
    'Cursed artifact manifestation',
    'Ancient burial ground disturbance',
    'Unexplained disappearance',
    'Betrayed by trusted friend',
    'Death during natural disaster'
]

for i in range(1, 21):
    base_date = datetime.now() - timedelta(days=np.random.randint(30, 365))
    last_date = base_date + timedelta(days=np.random.randint(1, 30))
    current_time = datetime.now()
    
    ghosts_data.append({
        'ghost_id': f'GH{str(i).zfill(3)}',
        'ghost_name': f'Specter #{i}',
        'ghost_type': np.random.choice(ghost_types),
        'threat_level': np.random.choice(threat_levels),
        'description': f'Paranormal entity #{i} detected in various locations',
        'manifestation_frequency': np.random.choice(frequencies),
        'origin_story': np.random.choice(origins),
        'first_detected_date': base_date,
        'last_seen_date': last_date,
        'status': 'Active',
        'confidence_score': round(np.random.uniform(0.6, 0.95), 2),
        'created_at': current_time,
        'updated_at': current_time
    })

ghosts_df = pd.DataFrame(ghosts_data)
print(f"✅ Created {len(ghosts_df)} ghost records with all required columns")
print(f"Columns ({len(ghosts_df.columns)}): {list(ghosts_df.columns)}")
ghosts_df.head()


In [ ]:
# Load ghosts to Snowflake
try:
    ghosts_sp_df = session.create_dataframe(ghosts_df)
    ghosts_sp_df.write.mode('append').save_as_table('GHOSTS')
    print(f"✅ Loaded {len(ghosts_df)} ghosts successfully")
except Exception as e:
    print(f"❌ Error loading ghosts: {str(e)}")


## 📥 Load 2: Sightings Data from CSV


In [ ]:
# Generate sample sightings data
np.random.seed(42)

locations = [
    ('Haunted Mansion', '123 Spooky Lane', 40.7589, -73.9851),
    ('Old Cemetery', '456 Graveyard Rd', 51.5074, -0.1278),
    ('Abandoned Hospital', '789 Dark Ave', 48.8566, 2.3522),
    ('Dark Forest', '321 Shadow Path', 35.6762, 139.6503),
    ('Ancient Castle', '654 Medieval St', 52.5200, 13.4050)
]

evidence_types = ['Visual', 'Audio', 'EMF', 'Temperature', 'Multiple']
conditions = ['Cold spot detected', 'EMF spike', 'Unusual sounds', 'Temperature drop', 'Visual anomaly']

sightings_data = []
current_time = datetime.now()
for i in range(100):
    loc = locations[i % len(locations)]
    lat = loc[2] + np.random.uniform(-0.01, 0.01)
    lon = loc[3] + np.random.uniform(-0.01, 0.01)
    
    sightings_data.append({
        'sighting_id': f'SIGHT{str(i).zfill(4)}',
        'ghost_id': f'GH{str(np.random.randint(1, 21)).zfill(3)}',
        'location_name': loc[0],
        'location_address': loc[1],
        'location_coordinates': f'POINT({lon} {lat})',
        'latitude': lat,
        'longitude': lon,
        'sighting_datetime': datetime.now() - timedelta(days=np.random.randint(0, 365)),
        'witness_name': f'Witness {i}',
        'witness_contact': f'witness{i}@example.com',
        'environmental_conditions': np.random.choice(conditions),
        'temperature_celsius': round(np.random.uniform(5, 25), 1),
        'emf_reading': round(np.random.uniform(0, 50), 2),
        'description': f'Sighting #{i} - Paranormal activity detected at {loc[0]}',
        'evidence_type': np.random.choice(evidence_types),
        'paranormal_activity_level': np.random.randint(1, 11),
        'investigation_notes': f'Investigation notes for sighting #{i}',
        'verified': np.random.choice([True, False]),
        'created_at': current_time
    })

sightings_df = pd.DataFrame(sightings_data)
print(f"✅ Created {len(sightings_df)} sighting records with all required columns")
print(f"Columns ({len(sightings_df.columns)}): {list(sightings_df.columns)}")
sightings_df.head()


In [ ]:
# Validate sightings data
invalid_records = []

for idx, row in sightings_df.iterrows():
    if not validate_coordinates(row['latitude'], row['longitude']):
        invalid_records.append(('invalid_coordinates', idx))
    if not validate_activity_level(row['paranormal_activity_level']):
        invalid_records.append(('invalid_activity', idx))
    if not validate_temperature(row['temperature_celsius']):
        invalid_records.append(('invalid_temperature', idx))
    if not validate_emf(row['emf_reading']):
        invalid_records.append(('invalid_emf', idx))

if invalid_records:
    print(f"⚠️ Found {len(invalid_records)} validation issues")
    print(invalid_records[:5])
else:
    print("✅ All sightings data validated successfully")


In [ ]:
# Load sightings to Snowflake
try:
    sightings_sp_df = session.create_dataframe(sightings_df)
    sightings_sp_df.write.mode('append').save_as_table('GHOST_SIGHTINGS')
    print(f"✅ Loaded {len(sightings_df)} sightings successfully")
except Exception as e:
    print(f"❌ Error loading sightings: {str(e)}")


## 📥 Load 3: Evidence & Images


In [ ]:
# Generate evidence records
evidence_types = ['Photo', 'Video', 'Audio', 'Sensor_Data']
mime_types = {
    'Photo': 'image/jpeg',
    'Video': 'video/mp4',
    'Audio': 'audio/wav',
    'Sensor_Data': 'application/json'
}

evidence_data = []
current_time = datetime.now()
for i in range(150):
    ev_type = np.random.choice(evidence_types)
    
    # Create sample metadata
    metadata = {
        'camera': 'Full Spectrum' if ev_type in ['Photo', 'Video'] else 'Audio Recorder',
        'exposure': '1/60s' if ev_type == 'Photo' else None,
        'iso': 3200 if ev_type == 'Photo' else None,
        'temperature': round(np.random.uniform(10, 25), 1),
        'emf': round(np.random.uniform(0, 50), 2)
    }
    
    evidence_data.append({
        'evidence_id': f'EV{str(i).zfill(4)}',
        'sighting_id': f'SIGHT{str(np.random.randint(0, 100)).zfill(4)}',
        'ghost_id': f'GH{str(np.random.randint(1, 21)).zfill(3)}',
        'evidence_type': ev_type,
        'file_path': f'@GHOST_DATA_STAGE/evidence/evidence_{i}.{ev_type.lower()}',
        'file_url': f'https://storage.example.com/evidence_{i}',
        'file_size_bytes': np.random.randint(100000, 10000000),
        'mime_type': mime_types[ev_type],
        'capture_datetime': datetime.now() - timedelta(days=np.random.randint(0, 365)),
        'image_data': None,  # Would contain base64 encoded image
        'thumbnail_data': None,  # Would contain base64 encoded thumbnail
        'metadata': metadata,  # Keep as dict for VARIANT type
        'processing_status': 'Pending',
        'created_at': current_time
    })

evidence_df = pd.DataFrame(evidence_data)
print(f"✅ Created {len(evidence_df)} evidence records with all required columns")
print(f"Columns ({len(evidence_df.columns)}): {list(evidence_df.columns)}")
evidence_df.head()


In [ ]:
# Load evidence to Snowflake
try:
    evidence_sp_df = session.create_dataframe(evidence_df)
    evidence_sp_df.write.mode('append').save_as_table('GHOST_EVIDENCE')
    print(f"✅ Loaded {len(evidence_df)} evidence records successfully")
except Exception as e:
    print(f"❌ Error loading evidence: {str(e)}")


## 📊 Data Quality Report


In [ ]:
# Query loaded data counts
counts = {}
tables = ['GHOSTS', 'GHOST_SIGHTINGS', 'GHOST_EVIDENCE', 'INVESTIGATORS', 'INVESTIGATIONS']

for table in tables:
    try:
        result = session.sql(f"SELECT COUNT(*) as cnt FROM {table}").collect()
        counts[table] = result[0]['CNT']
    except Exception as e:
        counts[table] = f"Error: {str(e)}"

print("\n📊 Data Quality Report")
print("=" * 50)
for table, count in counts.items():
    print(f"{table:25} {count:>20}")
print("=" * 50)


## 🔍 Data Validation Queries


In [ ]:
# Check for orphaned records
orphaned_sightings = session.sql("""
SELECT COUNT(*) as orphaned_count
FROM GHOST_SIGHTINGS s
LEFT JOIN GHOSTS g ON s.GHOST_ID = g.GHOST_ID
WHERE g.GHOST_ID IS NULL
""").collect()

print(f"Orphaned Sightings: {orphaned_sightings[0]['ORPHANED_COUNT']}")


In [ ]:
# Check coordinate validity
invalid_coords = session.sql("""
SELECT COUNT(*) as invalid_count
FROM GHOST_SIGHTINGS
WHERE LATITUDE NOT BETWEEN -90 AND 90
   OR LONGITUDE NOT BETWEEN -180 AND 180
""").collect()

print(f"Invalid Coordinates: {invalid_coords[0]['INVALID_COUNT']}")


In [ ]:
# Activity level distribution
activity_dist = session.sql("""
SELECT 
    PARANORMAL_ACTIVITY_LEVEL,
    COUNT(*) as count
FROM GHOST_SIGHTINGS
GROUP BY PARANORMAL_ACTIVITY_LEVEL
ORDER BY PARANORMAL_ACTIVITY_LEVEL
""").to_pandas()

print("\nActivity Level Distribution:")
print(activity_dist)


## 📈 Summary Statistics


In [ ]:
# Generate summary stats
summary_sql = """
SELECT 
    COUNT(DISTINCT ghost_id) as unique_ghosts,
    COUNT(DISTINCT location_name) as unique_locations,
    AVG(paranormal_activity_level) as avg_activity,
    AVG(temperature_celsius) as avg_temp,
    AVG(emf_reading) as avg_emf,
    SUM(CASE WHEN verified = TRUE THEN 1 ELSE 0 END) as verified_count,
    COUNT(*) as total_sightings
FROM GHOST_SIGHTINGS
"""

summary = session.sql(summary_sql).to_pandas()
print("\n📊 Sightings Summary Statistics:")
print(summary.T)


## 🎯 Completion Summary


In [ ]:
print("✅ Data Loading Complete!")
print("\nSummary:")
print(f"  - Ghosts loaded: {len(ghosts_df)}")
print(f"  - Sightings loaded: {len(sightings_df)}")
print(f"  - Evidence loaded: {len(evidence_df)}")
print(f"\nTotal records: {len(ghosts_df) + len(sightings_df) + len(evidence_df)}")
print("\nNext steps:")
print("  1. Review data quality report above")
print("  2. Check Streamlit app for visualization")
print("  3. Run analytics notebook for insights")
print("  4. Generate reports in Streamlit")


---

## 📝 Notes

**Data Loading Best Practices:**
1. Always validate data before loading
2. Use batch operations for large datasets
3. Check for orphaned records
4. Verify data quality after loading
5. Use stages for large file uploads

**For Production:**
- Use `scripts/bulk_ghost_processor.py` for large-scale loads
- Implement proper error handling and retry logic
- Add data lineage tracking
- Use Snowpipe for streaming data

**Next Steps:**
1. Run `notebooks/01_ghost_analytics.ipynb` for analysis
2. Check Streamlit app for visualization
3. Review data quality reports
4. Configure automated loading pipelines

**For Bulk Processing:**
```bash
# CSV import
python scripts/bulk_ghost_processor.py --mode csv --input data.csv

# Image batch
python scripts/bulk_ghost_processor.py --mode images --input /photos --ghost-id GH001

# JSON batch
python scripts/bulk_ghost_processor.py --mode json --input batch.json
```

---

**Based on:** https://github.com/tspannhw/AIM-Ghosts/blob/main/standardghostload.ipynb

**Version:** 2.1  
**Status:** ✅ Complete
